# M6. 골든 셋 평가 (06_m6_평가)

100문항 골든 셋(`data/golden/golden_set_100.csv`)으로 **검색 성능**(Hit@K, MRR@5)을 측정합니다.

| 구성 | 문항 수 |
|---|---|
| 건설업 사고사례 | 47 |
| 제조업 사고사례 | 37 |
| 사고사망 통계 | 16 |

- 질문 초안은 코덱스가 만들었고, **정답 ID와 정답 확인 문구는 이 노트북에서 실제 DB와 다시 대조**합니다.
- 코덱스 질문 중 Q06~Q85는 사례 본문(재해개요)을 거의 그대로 옮겨 쓴 문장이라 검색이 쉬워집니다. 그래서 같은 문항을 **현장 말투 질문으로 바꿔서 다시 측정**하고 두 결과를 비교합니다(6절).
- 사전 준비: 04 노트북으로 `data/chroma`에 사례 6,032건이 적재되어 있어야 하고, `.env`에 `OPENAI_API_KEY`가 있어야 합니다.

## 필요 패키지
이 노트북은 **pandas, chromadb, openai, python-dotenv** 를 사용합니다.
`ModuleNotFoundError`가 나면 터미널에서 설치한 뒤 커널을 재시작하세요.
```
uv add pandas chromadb openai python-dotenv
```

## 0. 준비

In [ ]:
import os, re, time
from pathlib import Path
import pandas as pd
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 120)

ROOT = Path.cwd()
while not (ROOT / "data" / "chroma").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "chroma").exists(), "data/chroma가 없습니다. 04 노트북을 먼저 실행하세요."

load_dotenv(ROOT / ".env")
assert os.getenv("OPENAI_API_KEY"), f"{ROOT / '.env'} 에 OPENAI_API_KEY=키값 을 넣고 커널을 재시작하세요."
client = OpenAI(timeout=60, max_retries=2)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")
K = 5   # 평가할 상위 순위

col = chromadb.PersistentClient(path=str(ROOT / "data" / "chroma")).get_collection("sif")
GOLD = ROOT / "data" / "golden" / "golden_set_100.csv"
A = ROOT / "data" / "analysis"; A.mkdir(parents=True, exist_ok=True)
assert GOLD.exists(), f"{GOLD} 가 없습니다. golden_set_100.csv를 data/golden 폴더에 넣으세요."
print("컬렉션 문서 수:", col.count())

## 1. 골든 셋 불러오기
컬럼: `ID`, `분야`, `질문`, `정답ID`(우리 DB의 문서 ID), `정답확인문구`.
- 사례 문항의 `정답ID`는 `제조업등-번호`, `건설업-번호` 형식으로 DB의 문서 ID와 같습니다.
- 통계 문항의 `정답ID`는 `통계-규모` 형식입니다. **현재 DB에는 통계 문서가 없습니다.**

In [ ]:
gold = pd.read_csv(GOLD, encoding="utf-8-sig")
gold["유형"] = gold["분야"].map(lambda s: "통계" if "통계" in s else "사례")
display(gold.groupby(["유형", "분야"]).size().to_frame("문항수"))
display(gold.head(3))

## 2. 정답 근거 검증 (정답 기준이 명확한가)
팀 기준은 "정답으로 잡은 기준이 명확해야 한다"입니다. 사례 문항마다
1. 정답 ID가 DB에 있는지,
2. 정답 확인 문구가 그 문서 본문에 실제로 들어 있는지
를 확인합니다. (문구를 의역한 문항은 `False`로 나오니 눈으로 확인합니다.)

In [ ]:
norm = lambda s: re.sub(r"\s+", "", str(s))
case = gold[gold["유형"] == "사례"].copy()
got = col.get(ids=list(case["정답ID"]))
doc_of = dict(zip(got["ids"], got["documents"]))
cause_of = {i: (m or {}).get("기인물", "") for i, m in zip(got["ids"], got["metadatas"])}   # 정답 사례의 기인물

case["DB에 존재"] = case["정답ID"].isin(doc_of)
case["문구 일치"] = [norm(p) in norm(doc_of.get(i, "")) for i, p in zip(case["정답ID"], case["정답확인문구"])]
print(f"사례 문항 {len(case)}개 | 정답 ID가 DB에 있음: {int(case['DB에 존재'].sum())} | 정답 문구가 본문에 있음: {int(case['문구 일치'].sum())}")
display(case[~case["문구 일치"]][["ID", "정답ID", "정답확인문구"]])

# 정답 ID 중복(같은 사례를 정답으로 쓰는 문항) 확인
print("정답 ID 중복 수:", int(case["정답ID"].duplicated().sum()))

## 3. 검색 성능 평가 함수
- 질문을 임베딩해 상위 5개 문서를 가져옵니다. (`MIN_SIM` 기준 없이 순위만 봅니다.)
- **Hit@K**: 상위 K개 안에 정답 문서가 있는 문항 비율
- **MRR@5**: 정답 문서의 순위 역수(1위=1, 2위=0.5, …, 5위 밖=0)의 평균
- **Precision@K**: 상위 K개 중 **관련 있는 문서의 비율**. 관련 문서 = 정답 문서 + **정답과 기인물이 같은 사례**. (같은 기인물 사례는 질문에 답하는 데 도움이 되기 때문입니다. 통계 문항은 정답 문서만 관련 문서로 봅니다.)

In [ ]:
def embed_many(texts, bs=100):
    out = []
    for s in range(0, len(texts), bs):
        r = client.embeddings.create(model=EMBED_MODEL, input=[t[:6000] for t in texts[s:s + bs]])
        out += [x.embedding for x in r.data]
    return out

def evaluate(df, qcol="질문", k=K):
    res = col.query(query_embeddings=embed_many(df[qcol].tolist()), n_results=k)
    rows = []
    for (_, r), ids, metas, dist in zip(df.iterrows(), res["ids"], res["metadatas"], res["distances"]):
        rank = ids.index(r["정답ID"]) + 1 if r["정답ID"] in ids else 0
        cause = cause_of.get(r["정답ID"], "")
        rel = [(i == r["정답ID"]) or (bool(cause) and (m or {}).get("기인물", "") == cause) for i, m in zip(ids, metas)]
        rows.append({"ID": r["ID"], "유형": r["유형"], "분야": r["분야"], "질문": r[qcol],
                     "정답ID": r["정답ID"], "정답순위": rank, "1위ID": ids[0], "1위유사도": round(1 - dist[0], 3),
                     "P@3": sum(rel[:3]) / 3, "P@5": sum(rel[:5]) / 5})
    return pd.DataFrame(rows)

def metrics(res, by=None):
    f = lambda d: pd.Series({
        "문항수": len(d),
        "Hit@1": round((d["정답순위"] == 1).mean(), 3),
        "Hit@3": round(d["정답순위"].between(1, 3).mean(), 3),
        "Hit@5": round(d["정답순위"].between(1, 5).mean(), 3),
        "MRR@5": round(d["정답순위"].map(lambda x: 1 / x if x else 0).mean(), 3),
        "Precision@3": round(d["P@3"].mean(), 3),
        "Precision@5": round(d["P@5"].mean(), 3),
    })
    return res.groupby(by).apply(f) if by else f(res).to_frame("전체").T

## 4. 코덱스 원본 질문 100문항 평가

In [ ]:
res_orig = evaluate(gold)
display(metrics(res_orig))
display(metrics(res_orig, by="유형"))
display(metrics(res_orig, by="분야"))
res_orig.to_csv(A / "m6_평가결과_원본질문.csv", index=False, encoding="utf-8-sig")

### 4-1. 못 찾은 문항
정답이 1위가 아닌 문항입니다. `정답순위 = 0`은 상위 5개 안에 없다는 뜻입니다.

In [ ]:
miss = res_orig[res_orig["정답순위"] != 1].sort_values(["유형", "정답순위"])
print("1위가 아닌 문항:", len(miss), "/", len(res_orig))
display(miss[["ID", "유형", "질문", "정답ID", "정답순위", "1위ID", "1위유사도"]])

### 4-2. 실패 사례 분석: 왜 1위가 아닌가
못 찾은 문항을 유형으로 나눕니다.
- **정답이 2~5위**: 정답은 가져왔지만 순위가 밀림
- **같은 기인물의 다른 사례가 1위**: 정답은 놓쳤지만 같은 종류의 사고 사례를 가져옴 (정답 기준이 좁아서 생기는 실패일 수 있음)
- **다른 기인물이 1위**: 엉뚱한 사례를 가져옴 (진짜 검색 실패)
- **통계**: 통계 문서가 인덱스에 없음

In [ ]:
top_meta = col.get(ids=list(miss["1위ID"].unique()))
cause_top = {i: (m or {}).get("기인물", "") for i, m in zip(top_meta["ids"], top_meta["metadatas"])}

def fail_kind(r):
    if r["유형"] == "통계":
        return "통계 (문서 없음)"
    if r["정답순위"] > 0:
        return "정답이 2~5위"
    same = cause_top.get(r["1위ID"]) and cause_top.get(r["1위ID"]) == cause_of.get(r["정답ID"])
    return "같은 기인물의 다른 사례가 1위" if same else "다른 기인물이 1위 (진짜 실패)"

miss = miss.copy()
miss["실패 유형"] = miss.apply(fail_kind, axis=1)
display(miss["실패 유형"].value_counts().to_frame("문항수"))
display(miss[miss["실패 유형"].str.contains("진짜")][["ID", "질문", "정답ID", "1위ID", "1위유사도"]])

### 4-3. 통계 문항은 왜 못 찾나
사고사망 통계 16문항의 정답은 `통계-…` 문서인데, 지금 DB(04 노트북)에는 **사고사례 6,032건만** 들어 있고 통계 문서가 없습니다.
검색 단계에서 정답이 나올 수 없는 구조입니다 → 개선 후보(M7): 통계 문서를 인덱스에 추가.

In [ ]:
stat = res_orig[res_orig["유형"] == "통계"]
print("통계 문항 Hit@5:", round(stat["정답순위"].between(1, 5).mean(), 3))
print("통계 질문 1위로 검색된 문서의 종류:", stat["1위ID"].str.split("-").str[0].value_counts().to_dict())
print("DB에 통계 문서가 있는가:", len(col.get(ids=["통계-5인미만"])["ids"]) > 0)

## 5. 질문 현실성 점검: 사례 본문을 그대로 쓴 질문인가
사례 문항 질문이 재해개요 문장을 그대로 포함하는지 확인합니다. 그대로 포함하면 검색이 지나치게 쉬워집니다.

In [ ]:
def overview_of(i):
    d = doc_of.get(i, "")
    return d.split("[재해개요]")[-1].split("\n[기인물]")[0]

def copied(q, i):
    ov = norm(overview_of(i))
    mid = ov[len(ov) // 3: len(ov) // 3 + 20]       # 재해개요 중간의 20자
    return len(mid) == 20 and mid in norm(q)

case["본문 복사"] = [copied(q, i) for q, i in zip(case["질문"], case["정답ID"])]
print(f"사례 문항 {len(case)}개 중 재해개요 문장이 그대로 들어간 질문: {int(case['본문 복사'].sum())}개")
display(case.groupby("본문 복사").size().to_frame("문항수"))

## 6. 현장 말투 질문으로 다시 평가
사례 문항을 LLM으로 **사고 상세(날짜·장소·수치)를 빼고 작업·기계·위험 상황의 핵심만 남긴 짧은 질문**으로 바꿔서 다시 측정합니다.
(변환 결과는 `data/golden/natural_questions.csv`에 저장되어 다시 실행해도 API를 부르지 않습니다.)
※ 현장 말투 질문은 비슷한 사례가 여러 건이라 정답 1건만 정답으로 보면 점수가 낮게 나올 수 있습니다. 절대값보다 **원본과의 차이**를 봅니다.

In [ ]:
NAT = ROOT / "data" / "golden" / "natural_questions.csv"
PROMPT = """다음은 산업재해 검색 평가용 질문입니다. 사고 상황 설명이 길게 들어 있습니다.
현장 작업자나 안전관리자가 실제로 물어볼 법한 짧은 질문 한 문장(40자 안팎)으로 바꿔 주세요.
- 날짜, 장소, 회사, 사람, 수치 등 사고 상세는 빼고 작업·기계·위험 상황의 핵심만 남깁니다.
- 원문 문장을 그대로 옮기지 않고 표현을 바꿉니다.
- 질문만 출력합니다.

원문: {q}"""

if NAT.exists():
    nat = pd.read_csv(NAT, encoding="utf-8-sig")
    print("저장된 변환 결과를 불러왔습니다:", len(nat), "건")
else:
    rows = []
    for n, (_, r) in enumerate(case.iterrows(), 1):
        out = client.chat.completions.create(
            model=CHAT_MODEL, temperature=0,
            messages=[{"role": "user", "content": PROMPT.format(q=r["질문"])}],
        ).choices[0].message.content.strip()
        rows.append({"ID": r["ID"], "현장질문": out})
        if n % 20 == 0: print(f"  변환 {n}/{len(case)}")
    nat = pd.DataFrame(rows)
    nat.to_csv(NAT, index=False, encoding="utf-8-sig")

gold_nat = case.merge(nat, on="ID")
display(gold_nat[["ID", "질문", "현장질문"]].head(5))

In [ ]:
res_nat = evaluate(gold_nat, qcol="현장질문")
cmp = pd.concat({
    "원본 질문": metrics(res_orig[res_orig["유형"] == "사례"]).iloc[0],
    "현장 말투 질문": metrics(res_nat).iloc[0],
}, axis=1).T
display(cmp)
res_nat.to_csv(A / "m6_평가결과_현장질문.csv", index=False, encoding="utf-8-sig")

## 7. 요약

In [ ]:
m_case = metrics(res_orig[res_orig["유형"] == "사례"]).iloc[0]
m_stat = metrics(res_orig[res_orig["유형"] == "통계"]).iloc[0]
m_nat = metrics(res_nat).iloc[0]
print(f"[원본 질문·사례 {int(m_case['문항수'])}문항] Hit@1 {m_case['Hit@1']:.2f} / Hit@5 {m_case['Hit@5']:.2f} / MRR@5 {m_case['MRR@5']:.2f} / Precision@5 {m_case['Precision@5']:.2f}")
print(f"[원본 질문·통계 {int(m_stat['문항수'])}문항] Hit@5 {m_stat['Hit@5']:.2f}  → 통계 문서가 인덱스에 없어 검색 불가")
print(f"[현장 말투 질문·사례 {int(m_nat['문항수'])}문항] Hit@1 {m_nat['Hit@1']:.2f} / Hit@5 {m_nat['Hit@5']:.2f} / MRR@5 {m_nat['MRR@5']:.2f} / Precision@5 {m_nat['Precision@5']:.2f}")
diff = m_case["Hit@5"] - m_nat["Hit@5"]
if diff > 0.05:
    print(f"→ 현장 말투로 바꾸면 Hit@5가 {diff:.2f} 낮아진다. 원본 점수는 본문을 베낀 질문 덕에 부풀려졌고, 실제 성능은 후자에 가깝다.")
else:
    print("→ 현장 말투로 바꿔도 점수 차이가 작다. 질문 표현에 강한 검색이다.")
print("→ M7 개선 후보: (1) 통계 문서를 인덱스에 추가 (2) 질문에서 기인물을 뽑아 메타데이터 필터로 사용")

## 정리
- 골든 셋 100문항의 정답 ID를 실제 DB와 대조해 검증했다 (정답 ID 존재, 정답 문구 일치).
- 검색 지표는 Hit@1/3/5, Precision@3/5, MRR@5로 측정했다. 사람 평가(정확성·근거성·출처 명확성 등)는 5점 척도로 별도 수행한다.
- 한계: 원본 질문은 사례 본문을 그대로 쓴 경우가 많아(5절) 점수가 높게 나올 수 있고, 통계 문서는 현재 인덱스에 없다(4-2절).
- 다음 단계(M7): 위 한계에서 나온 개선 방법 1개를 골라 같은 골든 셋으로 전후를 비교한다.